# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 1 — Exploratieve Data Analyse (EDA)** · Khaleel Al-Aqel

Dit notebook is het eerste deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`, `sample_submission.csv`) staat in
de map `data/`.

**Inhoud**

- 1.1 Imports en instellingen · 1.2 Data inladen · 1.3 Datatypes, basisstatistieken en verdeling
- 1.4 Datakwaliteit · 1.5 Tijdselementen toevoegen · 1.6 Patronen in de tijd
- 1.7 Decompositie: trend, seizoen en residu · 1.8 Relaties met `cnt` (juiste correlatiemaat)
- 1.9 Data opschonen · 1.10 Bevindingen van de EDA


# 1. Opdracht 1: Exploratieve Data Analyse (EDA)

## 1.1 Imports en instellingen


In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 9 – Time series analysis 1 (seasonal_decompose)

# Importeren van de packages die nodig zijn voor de EDA
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')


## 1.2 Data inladen

De klasse `DataLoader` leest de bestanden in en zet `date_hour` direct om naar het
**DateTime-formaat** (Radhakrishnan, 2023b), met `parse_dates` zoals in het notebook
*Regression Time Series 2* (De Haagse Hogeschool, z.d.-d). Het opschonen (methode `clean`) volgt
in 1.9; die code komt niet in de lesstof voor (Claude, 2026. Prompt 1: Portfolio 2 CityBike
notebook).

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (read_csv met parse_dates)
# Prompt 1: Portfolio 2 CityBike notebook (klasse DataLoader, clean)

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


loader = DataLoader(DATA_FOLDER)
train_raw = loader.load('train.csv')
test_raw = loader.load('test.csv')
sample_submission = loader.load('sample_submission.csv')
print('train:', train_raw.shape, '| test:', test_raw.shape,
      '| sample_submission:', sample_submission.shape)
train_raw.head()

## 1.3 Datatypes, basisstatistieken en verdeling

Zoals in de DataCamp-cursus van week 9 (Vincent, z.d., hoofdstuk 2) en opdracht 1 van
*Time series analysis 2* (De Haagse Hogeschool, z.d.-g) bekijken we eerst de datatypes, de
basisstatistieken en de verdeling van de target.

In [ ]:
# Bron: Week 10 – Time series analysis 2, opdracht 1 (info en describe)
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (Summary Statistics and Diagnostics)

# Datatypes en aantal niet-lege waarden per kolom
train_raw.info()

In [ ]:
# Bron: Week 10 – Time series analysis 2, opdracht 1 (info en describe)
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (Summary Statistics and Diagnostics)

# Basisstatistieken van iedere kolom: trainingsdata en testdata
display(train_raw.describe().round(3))
display(test_raw.describe().round(3))

In [ ]:
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (histogram en boxplot)

# Verdeling van de target (histogram en boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_raw['cnt'], bins=50, ax=axes[0])
axes[0].set(title='Verdeling van cnt', xlabel='cnt (fietsen per uur)')
sns.boxplot(x=train_raw['cnt'], ax=axes[1])
axes[1].set(title='Boxplot van cnt', xlabel='cnt (fietsen per uur)')
plt.tight_layout()
plt.show()

**Toelichting.**

- `date_hour` is een datum-tijd; alle andere kolommen zijn numeriek. Let op: `holiday` is
  **binair** (0/1) en `weathersit` is een **ordinale categorie** (1 = helder … 4 = zware regen),
  ook al zijn ze als getal opgeslagen. Dat bepaalt welke correlatiemaat we gebruiken (1.8) en dat
  we ze niet standaardiseren.
- `temp`, `atemp`, `hum` en `windspeed` zijn genormaliseerd (schaal 0 – 1).
- `cnt` loopt van 1 tot 977. Het gemiddelde (190) ligt ver boven de mediaan (143): de verdeling is
  **scheef naar rechts**, met veel rustige uren en weinig zeer drukke uren (de 'uitschieters' in de
  boxplot zijn gewone spitsuren, geen fouten).
- In de testperiode (december) is het gemiddeld kouder dan in de trainingsdata.

## 1.4 Datakwaliteit

Bij een tijdreeks moeten de tijdstappen **gelijkmatig verdeeld** zijn (Radhakrishnan, 2023b). We
controleren daarom ook op ontbrekende uren, naast missende waarden, dubbele uren en onmogelijke
waarden. De functie `check_quality` is gemaakt met hulp van GenAI (Claude, 2026. Prompt 1:
Portfolio 2 CityBike notebook).

In [ ]:
# Bron: Week 9 – slides Timeseries analyse 1 (tijdstappen gelijkmatig)
# Prompt 1: Portfolio 2 CityBike notebook (functie check_quality)

def check_quality(data):
    """Geef een overzicht van mogelijke datakwaliteitsproblemen."""
    full_range = pd.date_range(data['date_hour'].min(),
                               data['date_hour'].max(), freq='h')
    checks = {
        'missende waarden (totaal)': data.isna().sum().sum(),
        'dubbele uren': data['date_hour'].duplicated().sum(),
        'ontbrekende uren': len(full_range) - data['date_hour'].nunique(),
        'hum == 0': (data['hum'] == 0).sum(),
        'windspeed == 0': (data['windspeed'] == 0).sum(),
        'weathersit == 4': (data['weathersit'] == 4).sum(),
    }
    return pd.Series(checks)


display(pd.DataFrame({'train': check_quality(train_raw),
                      'test': check_quality(test_raw)}))

# Op welke dagen ontbreken de meeste uren?
all_hours = pd.date_range(train_raw['date_hour'].min(),
                          train_raw['date_hour'].max(), freq='h')
missing_hours = all_hours.difference(train_raw['date_hour'])
pd.Series(missing_hours.date).value_counts().head(5)

**Toelichting.**

- Er zijn **geen missende waarden** en **geen dubbele uren**.
- In de trainingsdata **ontbreken 163 uren**, vooral op een paar dagen (o.a. 29–30 oktober 2012 en
  eind januari 2011), waarschijnlijk door storingen of extreem weer. Voor de machine-learning modellen
  is dit geen probleem (elk uur wordt los voorspeld). Voor tijdreeksbewerkingen (FFT, ACF, SARIMAX)
  hebben we wél een regelmatige reeks nodig: daar vullen we ontbrekende uren op met interpolatie.
- **`hum == 0`** (22 keer) is een meetfout: 0% luchtvochtigheid komt in de praktijk niet voor. We
  vervangen deze waarden door lineaire interpolatie.
- **`windspeed == 0`** (±12%) laten we staan: windstilte kan echt voorkomen.
- **`weathersit == 4`** komt maar 3 keer voor; in modellen met categorieën voegen we 3 en 4 samen.
- De testdata is compleet (456 uren) en heeft geen kwaliteitsproblemen.

## 1.5 Tijdselementen toevoegen

Uit `date_hour` halen we tijdselementen, zoals in opdracht 4 van *Regression Time Series 1*
(De Haagse Hogeschool, z.d.-c) en in de seasonal plots van Kaggle-cursus les 3 (Holbrook, z.d.-d):
uur, dag van de week (getal en naam), weeknummer, maand, jaar en dag van het jaar. Daarnaast maken
we `workingday`: maandag t/m vrijdag en geen feestdag.

In [ ]:
# Bron: Week 9 – Regression Time Series 1, opdracht 4 (tijdselementen)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)

def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


eda = add_calendar_features(train_raw)
eda[['date_hour', 'hour', 'weekday', 'day_name', 'week', 'month', 'year',
     'dayofyear', 'workingday']].head(3)

## 1.6 Patronen in de tijd

Lijngrafieken zijn de beste manier om tijdreeksen te tonen (Radhakrishnan, 2023b; Vincent, z.d.).
We beginnen met de hele periode, met een voortschrijdend gemiddelde zoals in *Time series
analysis 1* (De Haagse Hogeschool, z.d.-f) en Kaggle-cursus les 2 (Holbrook, z.d.-f). Daarna
zoomen we in op het dag-, week- en jaarpatroon, met een *seasonal plot* zoals in Kaggle-cursus
les 3 (Holbrook, z.d.-d) en opdracht 5 van *Regression Time Series 1* (De Haagse Hogeschool,
z.d.-c).

In [ ]:
# Bron: Week 9 – Regression Time Series 1, opdracht 2 (resample)
# Bron: Week 9 – Time series analysis 1, opdracht 2 (moving average)
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)

# Gemiddelde verhuur per uur, per dag, met een voortschrijdend gemiddelde
daily_raw = eda.set_index('date_hour')['cnt'].resample('D').mean()

fig, ax = plt.subplots()
ax.plot(daily_raw, alpha=0.5, label='gemiddelde per dag')
ax.plot(daily_raw.rolling(30, center=True).mean(), color='red',
        label='30-daags voortschrijdend gemiddelde')
ax.set(title='Gemiddeld aantal verhuurde fietsen per uur (per dag)',
       ylabel='cnt per uur')
ax.legend()
plt.show()

In [ ]:
# Bron: Week 9 – Regression Time Series 1, opdracht 5 (patronen zoeken)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook):
#       seasonal plot met sns.lineplot

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
             'Saturday', 'Sunday']
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Dagpatroon per dag van de week
sns.lineplot(data=eda, x='hour', y='cnt', hue='day_name',
             hue_order=day_order, errorbar=None, ax=axes[0])
axes[0].set(title='Gemiddelde verhuur per uur, per dag van de week',
            ylabel='gemiddelde cnt')

# Jaarpatroon: gemiddelde per maand, per jaar
sns.barplot(data=eda, x='month', y='cnt', hue='year', errorbar=None,
            ax=axes[1])
axes[1].set(title='Gemiddelde verhuur per maand (0 = 2011, 1 = 2012)',
            ylabel='gemiddelde cnt')
plt.tight_layout()
plt.show()

**Toelichting.**

- **Trend:** in 2012 worden duidelijk meer fietsen verhuurd dan in 2011.
- **Jaarseizoen:** veel verhuur in de zomer en het vroege najaar, weinig in de winter. De testperiode
  (december) valt in het rustige seizoen; in november 2012 zakt de verhuur al.
- **Dag- en weekpatroon:** maandag t/m vrijdag hebben dezelfde vorm met twee spitsen (±8:00 en
  ±17:00–18:00, woon-werkverkeer). Zaterdag en zondag hebben één brede middagpiek (recreatie). Het
  dagpatroon hangt dus af van het **type dag** → interactie tussen uur en werkdag.
- **Uitschieter:** eind oktober 2012 is er een dag met bijna geen verhuur (zie 1.7 en 1.9).

Hoe hangt het weer samen met de verhuur? Met `melt()` zetten we de vier weervariabelen in één
kolom, zodat seaborn ze in één figuur naast elkaar kan tonen (tip uit opdracht 5 van
*Regression Time Series 1*; De Haagse Hogeschool, z.d.-c).

In [ ]:
# Bron: Week 9 – Regression Time Series 1, opdracht 5 (hint: melt())
# Bron: Week 9 – DataCamp Visualizing Time Series Data in Python,
#       hoofdstuk 2 (boxplot)

# Lange tabel: één kolom 'variabele' en één kolom 'waarde'
weather_long = eda.melt(id_vars='cnt',
                        value_vars=['temp', 'atemp', 'hum', 'windspeed'],
                        var_name='variabele', value_name='waarde')
weather_long['waarde'] = weather_long['waarde'].round(1)   # klassen van 0,1

grid = sns.relplot(data=weather_long, x='waarde', y='cnt', col='variabele',
                   kind='line', errorbar=None, height=3, aspect=1.2)
grid.set_axis_labels('waarde (genormaliseerd)', 'gemiddelde cnt')
plt.show()

# Weersituatie is een (ordinale) categorie: boxplot per categorie
sns.boxplot(data=eda, x='weathersit', y='cnt')
plt.title('Verhuur per weersituatie (1 = helder ... 4 = zware regen)')
plt.show()

**Toelichting.**

- **Temperatuur** (`temp`, `atemp`): hoe warmer, hoe meer verhuur, met een afvlakking bij de
  hoogste temperaturen.
- **Luchtvochtigheid:** vanaf ±0,3 daalt de verhuur bij hogere luchtvochtigheid (vochtig weer gaat
  vaak samen met regen). De zeldzame zeer lage waarden zijn deels meetfouten (`hum == 0`).
- **Windsnelheid:** weinig verband; alleen bij zeer harde wind (> 0,6) minder verhuur, maar daar
  zijn weinig metingen.
- **Weersituatie:** bij categorie 3 (lichte regen/sneeuw) daalt de verhuur sterk.

## 1.7 Decompositie: trend, seizoen en residu

Een tijdreeks kun je opsplitsen in **trend**, **seizoenscomponent** en **residu**:
$y_t = T_t + S_t + R_t$ (Radhakrishnan, 2023b; De Haagse Hogeschool, z.d.-f). We doen dit voor
het gemiddelde per dag met `seasonal_decompose`, met een periode van 7 dagen (het weekpatroon).
De trend is dan een **gecentreerd voortschrijdend gemiddelde** over 7 dagen: het window is gelijk
aan de periode van het seizoen. We controleren dat door de trend ook zelf uit te rekenen, zoals in
opdracht 2 van het notebook *Time series analysis 1* (De Haagse Hogeschool, z.d.-f).

In [ ]:
# Bron: Week 9 – Time series analysis 1, opdracht 1, 2 en 3
# Bron: Week 10 – Time series analysis 2, opdracht 2 (decompositie)

daily_filled = daily_raw.interpolate()          # een paar lege dagen opvullen
decomposition = seasonal_decompose(daily_filled, model='additive', period=7)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

# Controle: trend = gecentreerd voortschrijdend gemiddelde met window 7
manual_trend = daily_filled.rolling(window=7, center=True).mean()
difference = (manual_trend - decomposition.trend).abs().max()
print(f'Grootste verschil eigen trend vs statsmodels: {difference:.6f}')
print('Weekseizoen (gemiddelde afwijking per dag van de week):')
seasonal_week = decomposition.seasonal.groupby(
    decomposition.seasonal.index.day_name()).mean()
print(seasonal_week.reindex(day_order).round(1).to_string())

**Toelichting decompositie.**

- De zelf berekende trend (7-daags gecentreerd voortschrijdend gemiddelde) is precies gelijk aan die
  van `statsmodels` (verschil 0): `seasonal_decompose` berekent de trend met een moving average.
- **Trend:** groei van 2011 naar 2012, met een dip in de winter en een daling vanaf oktober 2012.
- **Weekseizoen:** klein, tussen −9 (zondag) en +6 (donderdag/vrijdag), terwijl er gemiddeld ±190
  fietsen per uur worden verhuurd. Het **totaal** per dag verschilt dus weinig tussen de dagen; het
  weekpatroon zit vooral in de **vorm** van de dag (spitsen op werkdagen, middagpiek in het weekend,
  zie 1.6). Daarom moeten we uur en type dag samen gebruiken.
- **Residu:** grote negatieve uitschieters op dagen met slecht weer. De grootste valt eind oktober
  2012 (zie 1.9).

## 1.8 Relaties met `cnt` (juiste correlatiemaat)

De Pearson-correlatie meet alleen **lineaire** verbanden tussen twee **numerieke** variabelen. Voor
andere soorten variabelen gebruiken we een passende maat (Radhakrishnan, 2025):

| Type variabele | Voorbeelden | Maat |
|---|---|---|
| numeriek | `temp`, `atemp`, `hum`, `windspeed` | Pearson $r$ (lineair) en Spearman $\rho$ (monotoon) |
| ordinaal | `weathersit` | Spearman $\rho$ (rangcorrelatie) |
| binair | `holiday`, `workingday` | punt-biseriële $r$ (= Pearson met een 0/1-variabele) |
| categorisch | `hour`, `weekday`, `month` | correlatieverhouding $\eta$ |

De **correlatieverhouding** $\eta$ meet hoeveel van de variantie van `cnt` verklaard wordt door de
categorieën (0 = niets, 1 = alles):
$$\eta^2 = \frac{\sum_{c} n_c(\bar{y}_c - \bar{y})^2}{\sum_i (y_i - \bar{y})^2}$$
De functie `correlation_ratio` staat niet in de lesstof en is gemaakt met hulp van GenAI (Claude,
2026. Prompt 1: Portfolio 2 CityBike notebook).

In [ ]:
# Bron: Week 8 – slides Veelvoorkomende fouten (correlatiemetrieken)
# Prompt 1: Portfolio 2 CityBike notebook (functie correlation_ratio)

def correlation_ratio(categories, values):
    """Correlatieverhouding eta tussen een categorische en numerieke kolom."""
    groups = values.groupby(categories)
    between = (groups.size() * (groups.mean() - values.mean()) ** 2).sum()
    total = ((values - values.mean()) ** 2).sum()
    return np.sqrt(between / total)


rows = []
for column in ['temp', 'atemp', 'hum', 'windspeed']:
    rows.append({'variabele': column, 'type': 'numeriek',
                 'Pearson r': eda[column].corr(eda['cnt']),
                 'Spearman rho': eda[column].corr(eda['cnt'],
                                                  method='spearman')})
rows.append({'variabele': 'weathersit', 'type': 'ordinaal',
             'Spearman rho': eda['weathersit'].corr(eda['cnt'],
                                                    method='spearman')})
for column in ['holiday', 'workingday']:
    rows.append({'variabele': column, 'type': 'binair',
                 'punt-biseriële r': eda[column].corr(eda['cnt'])})
for column in ['hour', 'weekday', 'month']:
    rows.append({'variabele': column, 'type': 'categorisch',
                 'eta': correlation_ratio(eda[column], eda['cnt'])})
pd.DataFrame(rows).set_index('variabele').round(3)

In [ ]:
# Bron: Week 8 – slides Veelvoorkomende fouten (correlatiematrix)

# Correlatie tussen de numerieke variabelen onderling (multicollineariteit)
numeric = ['cnt', 'temp', 'atemp', 'hum', 'windspeed']
sns.heatmap(eda[numeric].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0)
plt.title('Pearson-correlatie tussen de numerieke variabelen')
plt.show()

**Toelichting.**

- **`hour`** heeft veruit het sterkste verband ($\eta = 0{,}71$): het uur verklaart ±50% van de
  variantie ($\eta^2$). De Pearson-correlatie zou dit missen, want het verband is niet lineair (twee
  spitsen).
- **Temperatuur** ($r \approx 0{,}41$, $\rho \approx 0{,}43$), **luchtvochtigheid** ($r = -0{,}33$)
  en de **maand** ($\eta = 0{,}28$) hebben een duidelijk verband. **Weersituatie** heeft een zwak
  negatief rangverband ($\rho = -0{,}12$): regen (categorie 3) is zeldzaam, maar dan is het effect
  groot (boxplot in 1.6).
- `workingday` ($r = 0{,}03$) en `weekday` ($\eta = 0{,}03$) lijken **geen** verband te hebben. Toch
  zagen we in 1.6 dat het type dag het dagpatroon volledig verandert: werkdagen en weekenden hebben
  ongeveer evenveel verhuur per dag, maar op **andere uren**. Dat is een **interactie** met `hour`, en
  die ziet een correlatie tussen twee variabelen niet. Daarom verwijderen we variabelen **niet** op
  basis van een lage correlatie (Radhakrishnan, 2025).
- `temp` en `atemp` zijn bijna perfect gecorreleerd ($r = 0{,}99$): multicollineariteit. Voor bomen
  is dat geen probleem; bij lineaire regressie kan regularisatie helpen (3.3).

## 1.9 Data opschonen

We bekijken de dagen met de laagste gemiddelde verhuur. De code van deze paragraaf komt niet in de
lesstof voor (Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook).

In [ ]:
# Prompt 1: Portfolio 2 CityBike notebook (dagen met de laagste verhuur)

daily_raw.nsmallest(5).round(1)

Op **29 en 30 oktober 2012** is er bijna geen verhuur, en ontbreken veel uren. Ook in het residu
van de decompositie (1.7) is dit de grootste uitschieter. Dit is een uitzonderlijke gebeurtenis
(extreem weer) die niet representatief is voor normale dagen. De andere lage dagen worden wél
verklaard door slecht weer of feestdagen, en die kan een model leren. We verwijderen alleen deze
twee dagen uit de trainingsdata en passen de opschoning uit 1.4 toe (`hum == 0`).

In [ ]:
# Prompt 1: Portfolio 2 CityBike notebook (opschonen, zie 1.4 en 1.9)

OUTLIER_DAYS = ['2012-10-29', '2012-10-30']
train = loader.clean(train_raw, OUTLIER_DAYS)
test = loader.clean(test_raw, outlier_days=[])
print(f'train: {len(train_raw)} -> {len(train)} rijen | '
      f'test: {len(test)} rijen')
print('hum == 0 na opschonen:', (train['hum'] == 0).sum())

## 1.10 Bevindingen van de EDA

1. **Datakwaliteit:** geen missende waarden of dubbele uren. Wel 163 ontbrekende uren, 22 onmogelijke
   waarden voor `hum` (gecorrigeerd) en 2 uitzonderlijke dagen (verwijderd).
2. **Dagpatroon:** de grootste bron van variatie ($\eta = 0{,}71$). Op werkdagen twee spitsen
   (woon-werkverkeer), in het weekend één middagpiek. Uur en type dag moeten dus **samen** gebruikt
   worden (interactie).
3. **Trend:** in 2012 gemiddeld ±65% meer verhuur dan in 2011 (zie 2.3), met een daling eind 2012.
4. **Jaarseizoen:** veel verhuur in zomer en najaar, weinig in de winter. De testperiode (december)
   is rustig en koud.
5. **Weer:** warmer → meer verhuur; regen (`weathersit` 3) en hoge luchtvochtigheid → minder.
   Windsnelheid heeft weinig invloed.
6. **Target:** `cnt` is scheef verdeeld en de effecten werken als percentages (regen kost een deel van
   de verhuur). Een log-schaal of Poisson-loss past daarom goed (3.3 en 3.10).

## Bronnen (Opdracht 1)

Claude, 2026. Prompt 1: Portfolio 2 CityBike notebook. Taalmodel Claude Opus 5.5 (Anthropic),
gesprek van november 2026. *[link naar het gedeelde gesprek invullen]*

De Haagse Hogeschool. (z.d.-c). *Regression Time Series 1* [Jupyter-notebook, 26/27 Machine
Learning, week 9]. Brightspace.

De Haagse Hogeschool. (z.d.-d). *Regression Time Series 2* [Jupyter-notebook, 26/27 Machine
Learning, week 10 en 11]. Brightspace.

De Haagse Hogeschool. (z.d.-f). *Time series analysis 1: Trend and decomposition*
[Jupyter-notebook, 26/27 Machine Learning, week 9]. Brightspace.

De Haagse Hogeschool. (z.d.-g). *Time series analysis 2: Spectral analysis (Fourier Transform)*
[Jupyter-notebook, 26/27 Machine Learning, week 10]. Brightspace.

Holbrook, R. (z.d.-d). *Seasonality* [Kaggle-notebook; Kaggle-cursus Time Series, les 3]. Kaggle.
Geraadpleegd op 16 november 2026, van https://www.kaggle.com/code/ryanholbrook/seasonality

Holbrook, R. (z.d.-f). *Trend* [Kaggle-notebook; Kaggle-cursus Time Series, les 2]. Kaggle.
Geraadpleegd op 16 november 2026, van https://www.kaggle.com/code/ryanholbrook/trend

Radhakrishnan, V. M. (2023b). *Timeseries analyse 1 – Timeseries decomposition*
[PowerPoint-slides, week 9]. De Haagse Hogeschool.

Radhakrishnan, V. M. (2025). *Veelvoorkomende fouten* [PowerPoint-slides, week 8]. De Haagse
Hogeschool.

Vincent, T. (z.d.). *Visualizing time series data in Python* [Online cursus, week 9]. DataCamp.
https://www.datacamp.com/courses/visualizing-time-series-data-in-python
